# Phasing via a cost function

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

def show_calls(op, depth):
    graph = op.call_graph(max_depth=depth)
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

## `PhasingViaCostFunction`
Phases every basis state $|x\rangle$ by an amount proportional to a cost function $f(x)$

This Bloq implements a unitary $U_f(\gamma)$ which phases each computational state on which
the wave-function has support, by an amount proportional to a function of that computational
basis state. The general unitary can be defined as
$$
    U_f(\gamma) = \sum_{x=0}^{N-1} e^{i 2 \pi \gamma f(x)} |x\rangle \langle x|
$$

The strategy to implement $U_f(\gamma)$ is to use two oracles $O_\text{direct}$
and $O_\text{phase}$ s.t.
$$
U_f(\gamma) = O_\text{direct}^\dagger(\mathbb{I}\otimes O_\text{phase})O_\text{direct}
$$

$O^\text{direct}$ evaluates a $b_\text{direct}$-bit approximation of the cost function $f(x)$
and stores it in a new output cost register. Note that the cost register can represent
arbitrary fixed point values and be of type `QFxp(b_direct, n_frac, signed)`.
$$
O^\text{direct}|x\rangle|0\rangle^{\otimes b_\text{direct}}_\text{cost}=|x\rangle|f(x)\rangle
$$

$O^\text{phase}$ acts on the cost register computed by $O^\text{direct}$ and phases the
state $|f(x)\rangle$ by $e^{i 2\pi \gamma f(x)}$
$$
O^\text{phase}(\gamma)=\sum_{k=0}^{2^{b_\text{direct}}-1}e^{i 2\pi\gamma k}|k\rangle\langle k|
$$


Different strategies for implementing the two oracles would give different costs tradeoffs.
See `QvrZPow` and `QvrPhaseGradient` for two different implementations of
phase oracles described in the reference.

#### Parameters
 - `cost_eval_oracle`: Cost function evaluation oracle. Must compute the cost in a newly allocated RIGHT register.
 - `phase_oracle`: Oracle to phase the cost register. Must consume the cost register allocated by `cost_eval_oracle` as a THRU input. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix C: Oracles for phasing by cost function


In [ ]:
from examples.algorithms.square_cost_phasing_catalog import SquareCostPhasingExample

### Example Instances

In [ ]:
direct = SquareCostPhasingExample()

In [ ]:
gradient = SquareCostPhasingExample(via_gradient=True)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Square phase via gradient", gradient.signature, {}),
    ("Square phase via Z powers", direct.signature, {}),
])))

### Call Graph

In [ ]:
show_calls(gradient, depth=1)

In [ ]:
show_calls(direct, depth=None)

### Compile the two Square examples

The source examples both use five input bits and interpret the ten-bit square as an unsigned fraction. The native evaluator and its adjoint share the result and clean work wires. The gradient example uses a 32-bit caller-owned Fourier register. Prepare the negative-exponent gradient before applying the body for phase kickback. The following counts cover the body and report emitted gates separately from the source resource model.

In [ ]:
for op, measurements in ((direct, 40), (gradient, 409)):
    circuit = DynamicCircuit(op.qubit_count)
    circuit.append(op)
    counts = Counter(gate.name for gate in circuit.to_ir().instructions)
    model = estimate_component_resources(op.resource_model())
    print("Gradient width", op.gradient_width)
    print("Emitted on", op.qubit_count, "wires", dict(counts))
    print("Source logical resources", model)
    assert counts["measure"] == measurements